# 第一章：AI 安全实验环境与场景构建

本章是后续 AI 安全实验的起点：暂不训练模型，而是先把运行环境整理好——确认依赖、判断计算后端、建立工作目录，并保存一份环境报告方便后续排错。

后续章节的运行链路如下：

```text
输入数据 → 数据清洗与张量构造 → MindSpore 模型 → 计算后端 → 评估结果与模型权重
                                      │
                                      ├─ Ascend：MindSpore + CANN
                                      ├─ GPU：MindSpore GPU 后端
                                      └─ CPU：MindSpore CPU 后端
```

没有 Ascend 设备时，可先用 CPU/GPU 跑通流程，再迁移到 Ascend 完成完整训练。


## 一、实验场景与技术栈

- **Jupyter Notebook**
  - 在课程中的作用：组织讲解、代码与运行结果
  - 本章关注点：Kernel 是否使用了正确的 Python 环境
- **Python 环境**
  - 在课程中的作用：提供 NumPy、Pandas、Matplotlib 等依赖
  - 本章关注点：解释器位置与依赖是否可用
- **MindSpore**
  - 在课程中的作用：定义数据管道、网络、训练与推理流程
  - 本章关注点：能否导入并完成 Tensor、Dense 前向计算
- **CANN**
  - 在课程中的作用：为 Ascend 提供算子执行与运行时支持
  - 本章关注点：Ascend 环境下驱动和运行时是否匹配
- **Ascend / GPU / CPU**
  - 在课程中的作用：执行模型计算
  - 本章关注点：当前 Notebook 最终选择了哪个后端
- **CANNLab**
  - 在课程中的作用：提供云端开发环境和 AI 算力
  - 本章关注点：Notebook、Kernel 与计算资源是否正确连接

MindSpore 负责“要计算什么”，计算后端和运行时负责“在哪里、如何执行”。


## 二、课程目录约定

本课程采用扁平目录结构，所有 Notebook 直接放在课程根目录下，运行产物统一写入 `course_workspace/`，避免文件散落在项目根目录：

```text
01_threat_detect/
├── 01_environment_setup.ipynb
├── 02_ransomware_detection.ipynb
├── 03_encrypted_traffic_detection.ipynb
└── course_workspace/
    ├── data/          # 实验输入数据
    ├── outputs/       # 环境报告、运行结果
    ├── logs/          # 运行日志
    └── checkpoints/   # 模型权重
```

不要把报告、日志或临时文件散落在课程根目录。


## 三、安装依赖与 Kernel 准备

先在终端中准备依赖（本 Notebook 不自动安装，避免装到错误的环境）。确认 Python/pip 指向你要使用的环境后，按需安装缺失模块，安装完成后重启 Kernel。

Ascend 云环境优先使用平台预置的 MindSpore+Ascend 镜像，不要在 Notebook 内临时安装 MindSpore——它的版本需要和 Python、CANN、驱动、固件一起匹配。


In [10]:
# 一键安装本课程所需依赖：已安装的包跳过，缺失的包用 pip 自动安装。
import importlib.util
import subprocess
import sys

# 基础依赖：import 名 → pip 安装名
BASE_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}
MINDSPORE_PACKAGE = "mindspore"


def is_available(name):
    return importlib.util.find_spec(name) is not None


def pip_install(package):
    print(f"正在安装 {package} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    # 只打印末尾片段，避免输出过长
    print((result.stdout or "")[-1500:])
    return result.returncode == 0


print("=" * 70)
print("基础依赖检查与安装")
for import_name, install_name in BASE_PACKAGES.items():
    if is_available(import_name):
        print(f"{install_name:<12} 已安装，跳过")
    else:
        ok = pip_install(install_name)
        status = "安装成功" if ok and is_available(import_name) else "安装失败"
        print(f"{install_name:<12} {status}")

print("=" * 70)
print("MindSpore 检查")
if is_available(MINDSPORE_PACKAGE):
    import mindspore as ms
    print(f"mindspore 已安装，版本 {getattr(ms, '__version__', 'unknown')}，跳过")
else:
    print("未检测到 mindspore。")
    print("Ascend 云环境请使用平台预置的 MindSpore+Ascend 镜像，不要在此安装；")
    print("CPU 调试环境可取消下一行注释后重新运行本单元，安装 CPU 版 mindspore。")
    # pip_install("mindspore")

print("=" * 70)
print("安装完成。若刚安装了包，请重启 Notebook Kernel，再继续运行后续单元。")


基础依赖检查与安装
numpy        已安装，跳过
pandas       已安装，跳过
scikit-learn 已安装，跳过
matplotlib   已安装，跳过
openpyxl     已安装，跳过
MindSpore 检查
mindspore 已安装，版本 2.7.2，跳过
安装完成。若刚安装了包，请重启 Notebook Kernel，再继续运行后续单元。


## 四、导入依赖与全局配置

导入读取系统信息、处理路径、调用外部命令和保存报告所需的标准库，并封装一个统一的命令调用函数。


In [11]:
import os
import sys
import json
import platform
import subprocess
import importlib.util
from datetime import datetime
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    display = print


# 后面会多次调用系统命令，这里统一封装成一个小函数。
def run_command(command, timeout=8):
    try:
        completed = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=timeout,
            shell=False,
        )
        output = (completed.stdout or "").strip()
        return {
            "ok": completed.returncode == 0,
            "returncode": completed.returncode,
            "output": output[:2000],
        }
    except FileNotFoundError as exc:
        return {"ok": False, "returncode": None, "output": f"command not found: {exc}"}
    except Exception as exc:
        return {"ok": False, "returncode": None, "output": repr(exc)}


def show_package_status(rows):
    name_width = max(len("package"), *(len(row["package"]) for row in rows))
    print(f"{'package':<{name_width}}  available")
    print(f"{'-' * name_width}  ---------")
    for row in rows:
        print(f"{row['package']:<{name_width}}  {row['available']}")


设置统一的工作目录，后续实验的数据、日志、模型权重和环境报告都放在 `course_workspace/` 下。


In [12]:
# 当前 Notebook 所在目录
PROJECT_ROOT = Path.cwd().resolve()

# 实验数据和运行结果统一放在 course_workspace 下。
WORK_ROOT = PROJECT_ROOT / "course_workspace"
DATA_DIR = WORK_ROOT / "data"
OUTPUT_DIR = WORK_ROOT / "outputs"
LOG_DIR = WORK_ROOT / "logs"
CKPT_DIR = WORK_ROOT / "checkpoints"

# 后续会逐项检查这些依赖是否可用
REQUIRED_PACKAGES = [
    "mindspore",
    "numpy",
    "pandas",
    "sklearn",
    "matplotlib",
    "openpyxl",
]

# 环境检查结果会逐步写入 report，最后保存为 JSON。
report = {
    "experiment": "实验1：场景构建与环境配置",
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "project_root": str(PROJECT_ROOT),
    "notebook_location_hint": "课程 .ipynb 文件放在 project_root 下；数据放在 course_workspace/data/。",
    "work_root": str(WORK_ROOT),
}

print("项目根目录（请把课程 .ipynb 放在这里）：", PROJECT_ROOT)
print("课程工作目录（数据、输出、日志、checkpoint）：", WORK_ROOT)


项目根目录（请把课程 .ipynb 放在这里）： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect
课程工作目录（数据、输出、日志、checkpoint）： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace


## 五、CANN / Ascend 驱动检查

轻量检查 Notebook 能否看到 NPU 驱动。CPU/GPU 调试环境下 `npu-smi` 不可用可继续往下跑；是否真正能用 Ascend 训练，以第七节 MindSpore 自检的后端为准。


In [13]:
# 轻量检查 Ascend 驱动是否能被当前 Notebook Kernel 看到。
# 真正能否训练，以第七节 MindSpore 最小自检的后端结果为准。
npu_smi_result = run_command(["npu-smi", "info"], timeout=10)

cann_env = {
    "ASCEND_HOME_PATH": os.environ.get("ASCEND_HOME_PATH", ""),
    "ASCEND_VISIBLE_DEVICES": os.environ.get("ASCEND_VISIBLE_DEVICES", ""),
    "DEVICE_ID": os.environ.get("DEVICE_ID", ""),
}

report["cann_check"] = {
    "npu_smi_available": npu_smi_result["ok"],
    "npu_smi_output": npu_smi_result["output"],
    "env_hint": cann_env,
}

report["npu_smi"] = npu_smi_result
report["cann_env"] = [
    {"name": key, "configured": bool(value), "value_preview": value[:160]}
    for key, value in cann_env.items()
]

print("=" * 70)
print("Ascend 驱动轻量检查")
print("npu-smi 可用：", npu_smi_result["ok"])
if npu_smi_result["ok"]:
    print(npu_smi_result["output"][:1200])
else:
    print("未检测到 npu-smi；如果当前是 CPU/GPU 调试环境，可以继续运行后续单元。")
    print("需要 Ascend 训练时，请切换到预置 MindSpore+Ascend 的 ModelArts/Ascend 镜像后重试。")

print("环境提示：")
for key, value in cann_env.items():
    print(f"{key}: {value or '未设置'}")


Ascend 驱动轻量检查
npu-smi 可用： True
+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.5                   Version: 25.5.5                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip  Phy-ID              | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 5     Ascend910           | OK            | 179.8       45                0    / 0             |
| 0     10                  | 0000:0B:00.0  | 0           0    / 0          3571 / 65536         |
+------------------------------------------------------------------------------------------------+
| 5     Ascend910           | OK            | -           44                0 

## 六、本机与 Python 环境检查

记录 Python 版本与解释器路径，并检查课程依赖能否被当前 Kernel 找到。重点看 `python_executable` 是否正确、依赖的 `available` 是否为 `True`。


In [14]:
# 记录当前 Python 与系统信息。
system_info = {
    "python": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
}

# 检查课程依赖是否能被当前 Kernel 找到。
package_rows = []
for name in REQUIRED_PACKAGES:
    spec = importlib.util.find_spec(name)
    package_rows.append({"package": name, "available": spec is not None})

report["system_info"] = system_info
report["packages"] = package_rows

print("=" * 70)
print("系统信息")
for key, value in system_info.items():
    print(f"{key}: {value}")

print("=" * 70)
print("关键 Python 包可用性")
show_package_status(package_rows)


系统信息
python: 3.11.4 (main, Mar  9 2026, 01:22:03) [GCC 9.4.0]
python_executable: /opt/buildtools/Python-3.11.4/bin/python3.11
platform: Linux-5.10.0-182.0.0.95.r2220_156.hce2.aarch64-aarch64-with-glibc2.31
machine: aarch64
processor: aarch64
关键 Python 包可用性
package     available
----------  ---------
mindspore   True
numpy       True
pandas      True
sklearn     True
matplotlib  True
openpyxl    True


## 七、MindSpore 最小自检

导入 MindSpore，依次尝试 Ascend、GPU、CPU 后端，完成一个小 Tensor 运算和 Dense 前向计算。输出里的“当前可用后端”就是后续实验实际参考的结果。


In [15]:
# 记录 MindSpore 自检结果。
mindspore_check = {
    "available": False,
    "version": None,
    "selected_device": None,
    "tensor_check": None,
    "error": None,
}

try:
    import numpy as np
    import mindspore as ms
    import mindspore.nn as nn
    import mindspore.ops as ops
    from mindspore import Tensor

    mindspore_check["available"] = True
    mindspore_check["version"] = getattr(ms, "__version__", "unknown")

    # 优先尝试 Ascend，失败后再尝试 GPU 和 CPU。
    selected_device = None
    device_errors = {}
    for device in ["Ascend", "GPU", "CPU"]:
        try:
            ms.set_context(mode=ms.PYNATIVE_MODE, device_target=device)
            if device in {"Ascend", "GPU"}:
                try:
                    ms.set_context(device_id=int(os.environ.get("DEVICE_ID", "0")))
                except Exception:
                    pass

            # 用一个小 Tensor 验证当前后端能否执行算子。
            x = Tensor(np.ones((2, 3), dtype=np.float32))
            y = ops.ReduceSum()(x)
            selected_device = device
            mindspore_check["tensor_check"] = float(y.asnumpy())
            break
        except Exception as exc:
            device_errors[device] = repr(exc)

    mindspore_check["selected_device"] = selected_device
    mindspore_check["device_errors"] = device_errors

    if selected_device is None:
        raise RuntimeError(f"MindSpore 已安装，但 Ascend/GPU/CPU 后端均未通过自检：{device_errors}")

    # 再运行一个 Dense 层，确认 nn 模块可用。
    dense = nn.Dense(3, 2)
    out = dense(Tensor(np.ones((1, 3), dtype=np.float32)))
    mindspore_check["dense_output_shape"] = tuple(out.shape)

    print("=" * 70)
    print("MindSpore 版本：", mindspore_check["version"])
    print("当前可用后端：", selected_device)
    print("Tensor 求和自检：", mindspore_check["tensor_check"])
    print("Dense 输出形状：", mindspore_check["dense_output_shape"])

except Exception as exc:
    mindspore_check["error"] = repr(exc)
    print("MindSpore 自检未通过：", repr(exc))
    print("处理方式：检查 MindSpore 版本是否与当前 Python、CANN、驱动和硬件后端匹配。")

report["mindspore_check"] = mindspore_check


[WARNING] ME(11550:281472852406288,MainProcess):2026-08-05-21:42:47.894.000 [mindspore/run_check/_check_version.py:328] MindSpore version 2.7.2 and Ascend AI software package (Ascend Data Center Solution)version 9.0 does not match, the version of software package expect one of ['8.5']. Please refer to the match info on: https://www.mindspore.cn/install
[WARNING] ME(11550:281472852406288,MainProcess):2026-08-05-21:42:47.895.000 [mindspore/run_check/_check_version.py:346] MindSpore version 2.7.2 and "te" wheel package version 9.0 does not match. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(11550:281472852406288,MainProcess):2026-08-05-21:42:47.896.000 [mindspore/run_check/_check_version.py:359] Please pay attention to the above warning, countdown: 3
[WARNING] ME(11550:281472852406288,MainProcess):2026-08-05-21:42:48.896.000 [mindspore/run_check/_check_version.py:359] Please pay attention to the above warning, countdown: 2
[WARNING] ME(1

MindSpore 版本： 2.7.2
当前可用后端： Ascend
Tensor 求和自检： 6.0
Dense 输出形状： (1, 2)


## 八、创建工作目录并保存环境报告

创建后续实验所需目录（重复运行不清空已有文件），并把前面所有检查结果保存到 `course_workspace/outputs/environment_report.json`。后续实验出错时先看这份报告：检查 MindSpore 是否可导入、实际后端是什么、数据与 checkpoint 路径是否正确。


In [16]:
# 确保后续实验要用到的目录存在；重复运行不会清空已有文件。
workspace_paths = {
    "work_root": WORK_ROOT,
    "data_dir": DATA_DIR,
    "output_dir": OUTPUT_DIR,
    "log_dir": LOG_DIR,
    "checkpoint_dir": CKPT_DIR,
}

for path in workspace_paths.values():
    path.mkdir(parents=True, exist_ok=True)

report["workspace"] = {name: str(path) for name, path in workspace_paths.items()}

print("=" * 70)
print("课程工作目录已就绪")
print("数据目录：", DATA_DIR)
print("输出目录：", OUTPUT_DIR)
print("Checkpoint 目录：", CKPT_DIR)

# 汇总依赖状态，方便保存到环境报告中。
missing = [row["package"] for row in package_rows if not row["available"]]
report["missing_packages"] = missing
report["install_hint"] = "若存在缺失依赖，回到第一步，在终端执行注释中的 pip 安装/检测命令，并重启 Notebook Kernel。"

print("缺失依赖：", missing if missing else "无")
if missing:
    print("处理方式：回到第一步，在终端安装缺失依赖后重启 Notebook Kernel，再重新运行本 Notebook。")
else:
    print("基础依赖检查通过，可继续保存环境报告。")


课程工作目录已就绪
数据目录： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace/data
输出目录： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace/outputs
Checkpoint 目录： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace/checkpoints
缺失依赖： 无
基础依赖检查通过，可继续保存环境报告。


In [17]:
# 保存环境报告到 course_workspace/outputs/。
report_path = OUTPUT_DIR / "environment_report.json"

report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("环境报告已保存：", report_path)
print(json.dumps({
    "created_at": report["created_at"],
    "selected_device": report.get("mindspore_check", {}).get("selected_device"),
    "missing_packages": report.get("missing_packages", []),
    "report_path": str(report_path),
}, ensure_ascii=False, indent=2))


环境报告已保存： /mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace/outputs/environment_report.json
{
  "created_at": "2026-08-05 21:42:47",
  "selected_device": "Ascend",
  "missing_packages": [],
  "report_path": "/mnt/workspace/gitCode/cann-learning-hub/contrib/tutorials/ai_security_nuist/01_threat_detect/course_workspace/outputs/environment_report.json"
}


## 结论

环境已就绪：硬件驱动、Python 依赖、MindSpore 后端和工作目录均已确认，环境报告已保存。后续实验共用同一个 `course_workspace/`，每次结果都可追溯到对应的环境配置。
